In [1]:
!pip install langchain
!pip install langchain-community
!pip install langchain-google-genai
!pip install pypdf
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.9/565.9 kB 25.5 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
  

In [2]:
from langchain_community.document_loaders import PyPDFLoader

print("Libraries Loaded Successfully")

/tmp/ipykernel_485/2514659638.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Libraries Loaded Successfully


In [3]:
loader = PyPDFLoader("/content/cloud_computing_detailed_notes.pdf")

docs = loader.load()



print(len(docs))

30


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

print(len(chunks))

30


In [ ]:
#week 2 Embeddings and Vector Database

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
import os
from google.colab import userdata

os.environ["Gemini_api"] = userdata.get("Gemini_api")

In [6]:
import os

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    api_key=os.environ["Gemini_api"]
)

print("Embeddings Model Loaded Successfully")

Embeddings Model Loaded Successfully


In [7]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["Gemini_api"])

for model in genai.list_models():
    if "embedContent" in model.supported_generation_methods:
        print(model.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [8]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db"
)

print("Vector Database Created Successfully")

Vector Database Created Successfully


In [9]:
results = vector_db.similarity_search(
    "What is this document about?",
    k=2
)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}\n")
    print(doc.page_content[:400])


Result 1

Section 1: Cloud Computing Topic 1
Cloud computing is a modern technology that allows users to access computing resources over the
internet.
It eliminates the need for physical hardware and provides scalability, flexibility, and cost efficiency.
Organizations use cloud computing for storage, applications, and data processing.
1
Key concept explanation
2
Real-world example
3
Advantages and limitati

Result 2

Section 5: Cloud Computing Topic 5
Cloud computing is a modern technology that allows users to access computing resources over the
internet.
It eliminates the need for physical hardware and provides scalability, flexibility, and cost efficiency.
Organizations use cloud computing for storage, applications, and data processing.
1
Key concept explanation
2
Real-world example
3
Advantages and limitati


In [ ]:
# Week 3 - Retriever Setup

In [10]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("Retriever Created Successfully")

Retriever Created Successfully


In [11]:
query = "What is Cloud Computing?"

results = retriever.invoke(query)

print("Retrieved:", len(results), "documents")

Retrieved: 3 documents


In [12]:
for i, doc in enumerate(results, start=1):
    print(f"\nDocument {i}")
    print("-" * 50)
    print(doc.page_content)


Document 1
--------------------------------------------------
Section 23: Cloud Computing Topic 23
Cloud computing is a modern technology that allows users to access computing resources over the
internet.
It eliminates the need for physical hardware and provides scalability, flexibility, and cost efficiency.
Organizations use cloud computing for storage, applications, and data processing.
1
Key concept explanation
2
Real-world example
3
Advantages and limitations
4
Use cases in industry

Document 2
--------------------------------------------------
Section 25: Cloud Computing Topic 25
Cloud computing is a modern technology that allows users to access computing resources over the
internet.
It eliminates the need for physical hardware and provides scalability, flexibility, and cost efficiency.
Organizations use cloud computing for storage, applications, and data processing.
1
Key concept explanation
2
Real-world example
3
Advantages and limitations
4
Use cases in industry

Document 3
--

In [13]:
query = "Explain Virtualization"

results = retriever.invoke(query)

print(results[0].page_content)

Section 1: Cloud Computing Topic 1
Cloud computing is a modern technology that allows users to access computing resources over the
internet.
It eliminates the need for physical hardware and provides scalability, flexibility, and cost efficiency.
Organizations use cloud computing for storage, applications, and data processing.
1
Key concept explanation
2
Real-world example
3
Advantages and limitations
4
Use cases in industry


In [ ]:
# Week 4 - RAG Pipeline

In [ ]:
!pip install -U -q langchain langchain-core langchain-community langchain-google-genai

In [14]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

print("All imports successful")

All imports successful


In [15]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    google_api_key=os.environ["Gemini_api"]
)

print("Gemini LLM initialized successfully")

Gemini LLM initialized successfully


In [16]:
prompt = ChatPromptTemplate.from_template("""
You are an AI Research Assistant.

Answer the user's question using only the provided context.
If the answer is not available in the context, say:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{input}

Answer:
""")

print("RAG prompt created successfully")

RAG prompt created successfully


In [17]:
rag_chain = (
    {
        "context": retriever,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
)

print("RAG chain created successfully")

RAG chain created successfully


In [18]:
query = "What is Cloud Computing?"

response = rag_chain.invoke(query)

print(response.content)

Cloud computing is a modern technology that allows users to access computing resources over the internet. It eliminates the need for physical hardware and provides scalability, flexibility, and cost efficiency. Organizations use cloud computing for storage, applications, and data processing.


In [19]:
#week 5

In [20]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
    google_api_key=os.environ["Gemini_api"]
)

print("Gemini LLM initialized successfully")

Gemini LLM initialized successfully


In [21]:
prompt = ChatPromptTemplate.from_template("""
You are an AI Research Assistant.

Answer the user's question using only the provided context.

If the answer is not available in the context, say:
"I could not find the answer in the provided document."

Context:
{context}

Question:
{input}

Answer:
""")

print("RAG prompt created successfully")

RAG prompt created successfully


In [22]:
rag_chain = (
    {
        "context": retriever,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
)

print("RAG chain created successfully")

RAG chain created successfully


In [23]:
query = "What is Cloud Computing?"

response = rag_chain.invoke(query)

print("Answer:")
print(response.content)

Answer:
Cloud computing is a modern technology that allows users to access computing resources over the internet. It eliminates the need for physical hardware and provides scalability, flexibility, and cost efficiency. Organizations use cloud computing for storage, applications, and data processing.


In [24]:
query = "What are the advantages of cloud computing?"

retrieved_docs = retriever.invoke(query)

print("Retrieved Documents:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Retrieved Document {i} ---")
    print(doc.page_content[:300])

response = rag_chain.invoke(query)

print("\n--- Final Answer ---")
print(response.content)

Retrieved Documents: 3

--- Retrieved Document 1 ---
Section 3: Cloud Computing Topic 3
Cloud computing is a modern technology that allows users to access computing resources over the
internet.
It eliminates the need for physical hardware and provides scalability, flexibility, and cost efficiency.
Organizations use cloud computing for storage, applica

--- Retrieved Document 2 ---
Section 2: Cloud Computing Topic 2
Cloud computing is a modern technology that allows users to access computing resources over the
internet.
It eliminates the need for physical hardware and provides scalability, flexibility, and cost efficiency.
Organizations use cloud computing for storage, applica

--- Retrieved Document 3 ---
Section 1: Cloud Computing Topic 1
Cloud computing is a modern technology that allows users to access computing resources over the
internet.
It eliminates the need for physical hardware and provides scalability, flexibility, and cost efficiency.
Organizations use cloud computing for s

In [25]:
def ask_question(question):
    response = rag_chain.invoke(question)
    return response.content

In [26]:
answer = ask_question("What is virtualization?")
print(answer)

I could not find the answer in the provided document.
